[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/63_masked_perplexity_solution.ipynb)

# Solution: Masked Perplexity

Reference solution.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def masked_perplexity(logits, targets, mask):
    B, T, V = logits.shape
    logits = logits.reshape(B * T, V)
    targets = targets.reshape(B * T)
    mask = mask.reshape(B * T).to(logits.dtype)
    # stable log-softmax
    log_probs = logits - torch.logsumexp(logits, dim=-1, keepdim=True)
    # target log-prob per position, no torch.gather
    tgt_logp = log_probs[torch.arange(B * T), targets]
    nll = -(tgt_logp * mask).sum() / mask.sum().clamp_min(1.0)
    return torch.exp(nll)


In [ ]:
# Demo
B, T, V = 2, 5, 8
logits = torch.randn(B, T, V)
targets = torch.randint(0, V, (B, T))
mask = torch.tensor([[1, 1, 1, 0, 0], [1, 1, 1, 1, 0]], dtype=torch.float32)
print('PPL:', masked_perplexity(logits, targets, mask).item())


In [ ]:
from torch_judge import check
check('masked_perplexity')
